# Can forecast possession improve team tackle totals?

The first context experiment found that observed possession and territory reduced team tackle MAE from 26.51 to 18.85. Forecast context raised it to 30.50. This notebook tests a simpler question on held-out Six Nations rounds. Can we forecast the overall possession share, and can a tackle model use that forecast?

Fit the team, opponent and context effects together. A two-stage adjustment to a ratings forecast can count a team's usual possession pattern twice. Joint estimation lets the team and opponent coefficients adjust when possession enters the model. The tackle target is the sum of player-credited completed tackles across the full recorded team sheet.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from six_nations_fantasy.data.fantasy_fixtures import tournament_rounds
from six_nations_fantasy.data.lineup import match_context
from six_nations_fantasy.data.load import DATA_DIR, load_dataset
from six_nations_fantasy.models._share_regression import TeamRate
from six_nations_fantasy.models.player_share import full_lineups
from six_nations_fantasy.scraping.international import COMPETITIONS

years = [2024, 2025, 2026]
national_competitions = {label for _, label in COMPETITIONS.values()}
keys = ["match_id", "home"]
dataset = load_dataset(DATA_DIR)
context = pd.read_parquet(DATA_DIR / "team_performances.parquet")

## Construct a paired possession share

Sum the possession bins for each side. Across both sides the mass should be about 2.0. Divide each side's mass by the measured fixture total so the two resulting shares sum to one despite rounding. A side whose bins sum to 1.16 when the combined mass is 2.0 has 58% possession. Keep the territory bins in scraper order and check that the away vector reverses the home vector.

Use the same fixtures with complete context and two complete team sheets for every tackle variant, including the ratings baseline. This controls for sample differences. The coverage table reports the resulting exclusions.

In [ ]:
def context_columns(prefix):
    columns = context.filter(regex=rf"^{prefix}_[0-9]+$").columns
    if columns.empty:
        raise ValueError(f"No {prefix} columns in team_performances.parquet")
    return sorted(columns, key=lambda column: int(column.rsplit("_", 1)[1]))


possession_bins = context_columns("poss_percents")
territory_bins = context_columns("territory_percents")
bins = possession_bins + territory_bins
if not context["home"].isin([True, False]).all():
    raise ValueError("Invalid home indicator in team context")
paired = context.set_index(keys)[bins].unstack("home")
home, away = [
    paired.xs(side, level="home", axis=1)[bins] for side in (True, False)
]
complete = home.notna().all(axis=1) & away.notna().all(axis=1)
bounded = (home.ge(0) & home.le(1) & away.ge(0) & away.le(1)).all(axis=1)
combined_mass = (home[possession_bins] + away[possession_bins]).sum(axis=1, skipna=False)
possession_ok = combined_mass.sub(2).abs().le(.01 * len(possession_bins) + 1e-9)
territory_ok = np.isclose(home[territory_bins], away[territory_bins].to_numpy()[:, ::-1]).all(axis=1)
usable_context = complete & bounded & possession_ok & territory_ok

context = context.loc[context.match_id.isin(usable_context[usable_context].index)].copy()
context["possession"] = (
    context[possession_bins].sum(axis=1)
    / context.match_id.map(combined_mass)
)

lineup = match_context(dataset.performances, dataset.matches)
lineup["date"] = lineup.match_id.map(dataset.matches.set_index("match_id")["date"])
players, _, sides = full_lineups(lineup, training=True)
tackles = players["completed_tackles"].to_numpy(dtype=float)
if not np.isfinite(tackles).all() or (tackles < 0).any() or (tackles != np.floor(tackles)).any():
    raise ValueError("Expected complete, nonnegative integer tackle counts")

totals = players.groupby(keys)["completed_tackles"].sum().rename("tackles")
sides = sides.join(totals, on=keys)
sides = sides.merge(context[keys + ["possession"] + territory_bins], on=keys, validate="1:1")
sides = sides.loc[sides.groupby("match_id")["home"].transform("size").eq(2)]
print(f"{len(dataset.matches):,} fixtures, {len(sides) // 2:,} with usable context and complete sheets")

## Forecast possession from prior internationals

Use one observation per fixture. Fit home possession as a home-side intercept plus a shrunk possession rating for the home team minus the away team's rating. This assumes a team's tendency to keep the ball has equal and opposite effects on the sides' shares. An unseen team gets a zero rating. Compare with the mean home possession from the same prior internationals. Convert both forecasts to the away share by taking the complement.

Fit the tackle regressions on all eligible earlier club and international sides. The possession coefficient uses a ten-percentage-point unit. Territory uses the same unit after centering each bin on its training mean. Use a fixed prior scale of 0.5 on each continuous coefficient, and the existing opponent scale of 0.25 throughout. These values are set before scoring any round.

Score the joint possession model at the historical home mean as a control. Compare it with feeding in the matchup-specific possession forecast. This isolates the value of the possession forecast from the effect of refitting the tackle model. Observed possession and observed territory are retrospective diagnostics. The territory comparison asks what it adds given observed possession.

In [ ]:
def possession_forecasts(history, fixtures, penalty=5.0):
    # Return home possession under the mean and regularized matchup models.
    teams = pd.Index(pd.unique(history[["team", "opponent"]].to_numpy().ravel()))

    def contrasts(frame):
        return (
            frame["team"].to_numpy()[:, None] == teams.to_numpy()
        ).astype(float) - (frame["opponent"].to_numpy()[:, None] == teams.to_numpy())

    design = contrasts(history)
    center = design.mean(axis=0)
    centered = design - center
    home_mean = history["possession"].mean()
    effects = np.linalg.solve(
        centered.T @ centered + penalty * np.eye(len(teams)),
        centered.T @ (history["possession"].to_numpy() - home_mean),
    )
    matchup = home_mean + (contrasts(fixtures) - center) @ effects
    return pd.DataFrame(
        {"Historical home mean": home_mean, "Team and opponent": matchup.clip(0, 1)},
        index=fixtures["match_id"],
    )


def possession_feature(share):
    return (share - .5) * 10


def tackle_forecasts(training, validation, fixture_possession):
    territory_mean = training[territory_bins].mean()
    training = training.assign(possession_10pp=possession_feature(training["possession"]))
    validation = validation.assign(possession_10pp=possession_feature(validation["possession"]))
    training[territory_bins] = (training[territory_bins] - territory_mean) * 10
    validation[territory_bins] = (validation[territory_bins] - territory_mean) * 10
    counts = training["tackles"].to_numpy()
    possession_scale = {"possession_10pp": .5}

    ratings = TeamRate(training, counts, tau=.25)
    with_possession = TeamRate(training, counts, tau=.25, continuous_scales=possession_scale)
    with_territory = TeamRate(
        training, counts, tau=.25,
        continuous_scales={**possession_scale, **dict.fromkeys(territory_bins, .5)},
    )

    predictions = {
        "Ratings only": ratings.predict(validation),
        "Observed possession": with_possession.predict(validation),
        "Observed possession + territory": with_territory.predict(validation),
    }
    for model, column in [("Historical mean possession", "Historical home mean"),
                          ("Forecast possession", "Team and opponent")]:
        home_share = validation.match_id.map(fixture_possession[column])
        share = home_share.where(validation.home, 1 - home_share)
        predictions[model] = with_possession.predict(
            validation.assign(possession_10pp=possession_feature(share))
        )
    return predictions

## Walk forward by round

Every model is refit using dates strictly before the first scheduled game of the round. Score possession in percentage points once per fixture, and tackle totals once per side. Positive signed error means overprediction. Compare the MAE against the corresponding baseline and check consistency across seasons. Use the fixture-level predictions to inspect how errors differ within matches.

In [ ]:
coverage, possession_rows, tackle_rows = [], [], []
for year in years:
    for round_number, schedule in tournament_rounds(dataset.matches, year, price_free=True):
        training = sides.loc[sides.date.lt(schedule.date.min())]
        validation = sides.loc[sides.match_id.isin(schedule.match_id)]
        international_home = training.loc[
            training.home & training.competition.isin(national_competitions)
        ]
        home_fixtures = validation.loc[validation.home]
        coverage.append({
            "year": year, "round": round_number, "scheduled": len(schedule),
            "evaluated": len(home_fixtures) if len(international_home) else 0,
            "prior international fixtures": len(international_home),
            "training sides": len(training),
        })
        if home_fixtures.empty or international_home.empty:
            continue

        fixture_possession = possession_forecasts(international_home, home_fixtures)
        observed_home = home_fixtures.set_index("match_id")["possession"]
        for label, predictions in fixture_possession.items():
            possession_rows.append(pd.DataFrame({
                "year": year, "round": round_number, "match_id": predictions.index,
                "model": label, "observed_pp": 100 * observed_home.reindex(predictions.index),
                "predicted_pp": 100 * predictions,
            }).reset_index(drop=True))

        forecasts = tackle_forecasts(training, validation, fixture_possession)
        for label, predictions in forecasts.items():
            tackle_rows.append(validation[keys + ["team", "tackles"]].assign(
                year=year, round=round_number, model=label, predicted=predictions,
            ))

display(pd.DataFrame(coverage))
if not tackle_rows:
    raise ValueError("No evaluable rounds with earlier international possession history")

possession_results = pd.concat(possession_rows, ignore_index=True)
possession_results["error"] = possession_results.predicted_pp - possession_results.observed_pp
tackle_results = pd.concat(tackle_rows, ignore_index=True)
tackle_results["error"] = tackle_results.predicted - tackle_results.tackles


def error_summary(results, groups, baseline):
    summary = results.assign(absolute_error=results.error.abs()).groupby(groups).agg(
        observations=("error", "size"),
        MAE=("absolute_error", "mean"),
        signed_error=("error", "mean"),
    )
    baseline_mae = summary.xs(baseline, level="model")["MAE"] if len(groups) > 1 else summary.loc[baseline, "MAE"]
    summary["MAE_vs_baseline"] = summary["MAE"] - baseline_mae
    return summary.round(2)


print("Home possession (percentage points)")
display(error_summary(possession_results, ["model"], "Historical home mean"))
display(error_summary(possession_results, ["year", "model"], "Historical home mean"))

print("Team completed tackles")
display(error_summary(tackle_results, ["model"], "Ratings only"))
display(error_summary(tackle_results, ["year", "model"], "Ratings only"))

territory = tackle_results.pivot(index=keys, columns="model", values="error").abs()
gain = territory["Observed possession"] - territory["Observed possession + territory"]
print(f"Incremental MAE reduction from observed territory: {gain.mean():.2f} tackles")

If observed possession helps and the matchup forecast has little advantage over the home mean, the strongest evidence is for possession as an explanation of volume after the fact. If the matchup forecast beats the home mean on possession but still fails to help tackle totals, compare the two possession inputs to the same fitted tackle regression. That comparison shows whether better possession forecasts translate into useful differences in tackle forecasts. Signed error helps identify systematic overprediction or underprediction. The observed-territory row measures the value of the territory bins under hindsight only.